# Seed Robustness — BanglaBERT (Kaggle)

Runs **Cell 9** of `training.ipynb` on a Kaggle GPU. The Cell 9 code is copied verbatim; only the paths change.

**Before running**
1. Right panel → **Add Input** → your private dataset `cyberbullying-splits` (made from `kaggle_bundle.zip`)
2. Settings → **Accelerator: GPU T4 x2** (same GPU type as the Colab seed-42 run)
3. Settings → **Internet: On** (needed for the BanglaBERT download and the normalizer)
4. **Save Version → Save & Run All (Commit)**. It runs in the background, ~25 min, and keeps going if you close the tab

**After it finishes:** Output tab → download `seed_outputs.zip` → put the CSVs into Drive `reports/results/`.

In [ ]:
# ============================================================
# KAGGLE SETUP — rebuild the Drive folder layout inside /kaggle/working
# Kaggle inputs are read-only, so the needed files are copied into a
# writable layout. Cell 9 then runs unchanged against SPLIT_DIR / RESULT_DIR.
# ============================================================
import glob, os, shutil, json
import pandas as pd, torch, transformers

SPLIT_DIR  = '/kaggle/working/splits'
RESULT_DIR = '/kaggle/working/results'
os.makedirs(SPLIT_DIR, exist_ok=True); os.makedirs(RESULT_DIR, exist_ok=True)

NEEDED = {
    SPLIT_DIR:  ['train.csv', 'val.csv', 'test.csv', 'label_map.json', 'class_weights.npy'],
    RESULT_DIR: ['test_train_similarity.npy', 'tfidf_test_predictions.csv',
                 'banglabert_test_predictions.csv', 'banglabert_metrics.json',
                 'comparison_summary.csv'],
}
for dest, names in NEEDED.items():
    for name in names:
        hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
        assert len(hits) == 1, f'{name}: expected 1 match under /kaggle/input, found {hits} — dataset attached?'
        shutil.copy(hits[0], f'{dest}/{name}')
print('input files copied')

train = pd.read_csv(f'{SPLIT_DIR}/train.csv')
val   = pd.read_csv(f'{SPLIT_DIR}/val.csv')
test  = pd.read_csv(f'{SPLIT_DIR}/test.csv')
lm = json.load(open(f'{SPLIT_DIR}/label_map.json'))['label2id']
LABELS = [l for l, _ in sorted(lm.items(), key=lambda kv: kv[1])]
print(len(train), len(val), len(test), LABELS)          # expect 28918 6197 6197
assert (len(train), len(val), len(test)) == (28918, 6197, 6197), 'split sizes differ from the Colab run'

assert torch.cuda.is_available(), 'No GPU — Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| transformers', transformers.__version__)

!pip install -q git+https://github.com/csebuetnlp/normalizer
from normalizer import normalize
print('normalizer OK')

In [ ]:
# ============================================================
# CELL 9 — BanglaBERT Seed Robustness
# Cell 8's significance test covers test-sample noise, not training
# noise: Cell 7 was ONE fine-tuning run. Two more seeds with the
# identical config show whether +0.022 survives a different seed.
# Needs GPU. ~20 min on T4. Needs Cell 1 + Cell 6 in this session.
# ============================================================
import json, time, random, gc, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from scipy.stats import binomtest

try:
    train, val, test, LABELS, SPLIT_DIR, RESULT_DIR
except NameError:
    raise RuntimeError('Run Cell 1 and Cell 6 first (they load the splits and paths).')

MODEL_NAME = 'csebuetnlp/banglabert'
MAX_LEN, BATCH, EPOCHS, LR = 128, 32, 3, 2e-5      # identical to Cell 7
NEW_SEEDS = [13, 87]                               # seed 42 already trained in Cell 7
device = 'cuda'
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'

# free the Cell 7 model if it is still on the GPU
for _v in ('model', 'opt', 'sched', 'scaler', 'best_state'):
    globals().pop(_v, None)
gc.collect(); torch.cuda.empty_cache()

try:
    from normalizer import normalize as bn_normalize
except Exception:
    raise RuntimeError('csebuetnlp normalizer missing — Cell 7 used it, seeds must match. '
                       'Run: !pip install -q git+https://github.com/csebuetnlp/normalizer')

# ---- tokenize once up front (faster than per-item tokenization) ----
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
def encode(df):
    enc = tok([bn_normalize(str(t)) for t in df['text_bert']], truncation=True,
              max_length=MAX_LEN, padding='max_length', return_tensors='pt')
    return TensorDataset(enc['input_ids'], enc['attention_mask'],
                         torch.tensor(df['label_id'].values, dtype=torch.long))
ds_train, ds_val, ds_test = encode(train), encode(val), encode(test)

cw     = torch.tensor(np.load(f'{SPLIT_DIR}/class_weights.npy'), dtype=torch.float, device=device)
clean  = np.load(f'{RESULT_DIR}/test_train_similarity.npy') < 0.80
y_val  = val['label_id'].values
y_test = test['label_id'].values
p_tf   = pd.read_csv(f'{RESULT_DIR}/tfidf_test_predictions.csv')['pred_id'].values

@torch.no_grad()
def predict_with(model, ds):
    model.eval(); out = []
    for ids, am, _ in DataLoader(ds, batch_size=BATCH * 2):
        with torch.amp.autocast('cuda'):
            logits = model(input_ids=ids.to(device), attention_mask=am.to(device)).logits
        out.append(logits.float().argmax(-1).cpu())
    return torch.cat(out).numpy()

def run_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)   # also fixes classifier-head init
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS)).to(device)
    dl = DataLoader(ds_train, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True,
                    generator=torch.Generator().manual_seed(seed))
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps  = len(dl) * EPOCHS
    sched  = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler('cuda')
    loss_fn = nn.CrossEntropyLoss(weight=cw)

    best_f1, best_state = -1, None
    for ep in range(1, EPOCHS + 1):
        model.train(); t0, running = time.time(), 0.0
        for ids, am, yb in dl:
            ids, am, yb = ids.to(device), am.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = loss_fn(model(input_ids=ids, attention_mask=am).logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item()
        vf1 = f1_score(y_val, predict_with(model, ds_val), average='macro')
        print(f'  seed {seed} epoch {ep}: loss={running/len(dl):.4f}  '
              f'val macro-F1={vf1:.4f}  ({time.time()-t0:.0f}s)')
        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    pt = predict_with(model, ds_test)
    del model, opt, sched, best_state; gc.collect(); torch.cuda.empty_cache()
    return pt, best_f1

def seed_row(seed, pt, val_f1):
    ok_tf, ok_bb = p_tf == y_test, pt == y_test
    only_tf, only_bb = int((ok_tf & ~ok_bb).sum()), int((~ok_tf & ok_bb).sum())
    per = f1_score(y_test, pt, average=None, labels=range(len(LABELS)))
    return {'seed': seed, 'val_macro_f1': val_f1,
            'test_macro_f1':  f1_score(y_test, pt, average='macro'),
            'clean_macro_f1': f1_score(y_test[clean], pt[clean], average='macro'),
            'test_accuracy':  float(ok_bb.mean()),
            **{f'f1 {l}': f for l, f in zip(LABELS, per)},
            'mcnemar_p_vs_tfidf': binomtest(only_tf, only_tf + only_bb, 0.5).pvalue}

# ------------------------------------------------------------
# 1. Seed 42 from Cell 7 + the new seeds
# ------------------------------------------------------------
bb42 = pd.read_csv(f'{RESULT_DIR}/banglabert_test_predictions.csv')
assert (bb42['Text'].values == test['Text'].values).all(), 'seed-42 predictions not aligned with test'
m42  = json.load(open(f'{RESULT_DIR}/banglabert_metrics.json'))
rows = [seed_row(42, bb42['pred_id'].values, m42['config']['best_val_macro_f1'])]

for s in NEW_SEEDS:
    print(f'\n=== training seed {s} ===')
    pt, vf1 = run_seed(s)
    pd.DataFrame({'Text': test['Text'], 'label_id': y_test, 'pred_id': pt}) \
      .to_csv(f'{RESULT_DIR}/banglabert_test_predictions_seed{s}.csv', index=False)
    rows.append(seed_row(s, pt, vf1))

res = pd.DataFrame(rows).set_index('seed')
print('\n' + '=' * 60); print('PER-SEED RESULTS')
print(res.round(4).T.to_string())

# ------------------------------------------------------------
# 2. Mean ± std vs TF-IDF (deterministic, so no seed spread)
# ------------------------------------------------------------
tfs = pd.read_csv(f'{RESULT_DIR}/comparison_summary.csv', index_col='model').loc['TF-IDF + LogReg']
col_map = {'test_macro_f1': 'macro_f1', 'clean_macro_f1': 'macro_f1_clean',
           'test_accuracy': 'accuracy', **{f'f1 {l}': f'f1 {l}' for l in LABELS}}
agg = res[list(col_map)].agg(['mean', 'std']).T
agg['tfidf'] = [tfs[col_map[c]] for c in agg.index]
agg['delta_mean'] = agg['mean'] - agg['tfidf']
print('\n' + '=' * 60); print('BanglaBERT (3 seeds) vs TF-IDF')
print(agg.round(4).to_string())

m, sd = agg.loc['test_macro_f1', 'mean'], agg.loc['test_macro_f1', 'std']
all_beat = (res['test_macro_f1'] > tfs['macro_f1']).all()
all_sig  = (res['mcnemar_p_vs_tfidf'] < 0.05).all()
print(f'\nBanglaBERT test macro-F1 : {m:.4f} ± {sd:.4f}  (n=3 seeds)')
print(f'TF-IDF test macro-F1     : {tfs["macro_f1"]:.4f}')
print(f'beats TF-IDF in every seed          : {all_beat}  (worst seed {res["test_macro_f1"].min():.4f})')
print(f'McNemar p < 0.05 in every seed      : {all_sig}')
print('-> ROBUST' if all_beat and all_sig else '-> NOT robust across seeds — report with caution')

res.round(4).to_csv(f'{RESULT_DIR}/banglabert_seed_results.csv')
agg.round(4).to_csv(f'{RESULT_DIR}/banglabert_seed_summary.csv')
print(f'\nSaved -> banglabert_seed_results.csv | banglabert_seed_summary.csv | '
      f'banglabert_test_predictions_seed{{13,87}}.csv')

In [ ]:
# ============================================================
# Package the NEW files for download (Output tab -> seed_outputs.zip)
# ============================================================
import zipfile
new_files = ['banglabert_seed_results.csv', 'banglabert_seed_summary.csv',
             'banglabert_test_predictions_seed13.csv', 'banglabert_test_predictions_seed87.csv']
with zipfile.ZipFile('/kaggle/working/seed_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in new_files:
        z.write(f'{RESULT_DIR}/{f}', arcname=f)
print('seed_outputs.zip ->', new_files)